1. Install the Vertex AI SDK: Open a terminal window and enter the command below. You can also [install it in a virtualenv](https://googleapis.dev/python/aiplatform/latest/index.html)

In [ ]:
import vertexai

vertexai.init(project="rhazes-research", location="us-central1")

In [ ]:
from google.colab import auth

auth.authenticate_user()

In [ ]:
from google.auth import default, transport

credentials, _ = default()
auth_request = transport.requests.Request()
credentials.refresh(auth_request)

In [ ]:
import openai

MODEL_LOCATION = "us-east5"
MAAS_ENDPOINT = f"{MODEL_LOCATION}-aiplatform.googleapis.com"

client = openai.AsyncOpenAI(
    base_url=f"https://{MAAS_ENDPOINT}/v1beta1/projects/rhazes-research/locations/us-east5/endpoints/openapi",
    api_key=credentials.token,
)

In [ ]:
MODEL_ID = "meta/llama-4-scout-17b-16e-instruct-maas"

In [ ]:
max_tokens = 4096

response = await client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {"text": "What model are you?", "type": "text"},
            ],
        },
    ],
    max_tokens=max_tokens,
)

In [ ]:
response.choices[0].message.content

"I'm based on Llama, a model designed by Meta."

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

global result_df, prompt_no_example

result_df = pd.read_csv('/content/drive/MyDrive/MIMIC-IV_input.csv', index_col=0)

In [ ]:
prompt_no_example = '''You are an expert diagnostician machine for use by doctors. If the user input is not patient data, you politely decline the request. Please suggest diagnoses and conditions, followed by the evidence points supporting each diagnosis in the form of bullet points. Include previous diagnoses and pertinent information about the patient's medical history (if any). Pay close attention to all the history and investigations provided.  Put asterisks around the diagnoses to highlight them. Give each evidence points as a separate bullet point beneath the diagnosis. Include in your evidence points any relevant clinical scores that can be calculated from the information I have given. Do not explain the evidence points, only state them. For every diagnosis you list, if there are alternative differentials possible, state the most likely three in a bullet point beneath the evidence points (you do not need to state the evidence supporting them - you only need to do that for the main diagnoses). For the main diagnoses, give only confirmed diagnoses and evidence points that can be inferred solely based on the information I have given - do not use any other information. Only give me the information I have asked for - do not give me any other information. Do not give me any introductions or conclusions, safety instructions, or safety warnings. Use British English.

                                To illustrate how the information should be presented:

                                *MAIN DIAGNOSIS 1 AS HEADING*
                                evidence points to support MAIN DIAGNOSIS 1
                                The final bullet point is alternative differentials to consider: alternative 1, alternative 2, alternative 3

                                *MAIN DIAGNOSIS 2 AS HEADING*
                                evidence points to support MAIN DIAGNOSIS 2
                                The final bullet point is alternative differentials to consider: alternative 1, alternative 2, alternative 3

                                and so on...

Before finalising your answer check if you haven't missed any abnormal data points and hence any diagnoses or alternative differentials that could be made based on them. If you did, add them to your reply. If two diagnoses are commonly caused by the same underlying disease, have them under one header, which is the underlying disease.

Patient data:\n'''


In [ ]:
import asyncio
from google import genai
from google.genai import types
import pandas as pd
from tqdm import tqdm
import nest_asyncio

nest_asyncio.apply()

async def generate(prompt):
    response = await client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {"text": prompt, "type": "text"},
            ],
        },
    ],
    max_tokens=max_tokens,
    )

    return response.choices[0].message.content

async def process_row(i):
    response = await generate(  # Await the generate function
        prompt_no_example + result_df.iloc[i]['GPT_input']
    )
    hadm_id = result_df.index[i]
    return hadm_id, response # Return both hadm_id and the response

async def get_diagnoses(indices):
    tasks = []
    for i in indices:
        tasks.append(process_row(i))
    results = await asyncio.gather(*tasks)
    return results

In [ ]:
result_df['GPT-Diagnoses'] = result_df['GPT-Diagnoses'].astype('str')
result_df['GPT-Eval'] = result_df['GPT-Eval'].astype('str')
repeat = []

limit = 1000
chunk_size = 5

def generate_indices(limit, chunk_size):
  """Generates lists of indices in chunks."""
  start = 0
  while start < limit:
    end = min(start + chunk_size, limit)  # Ensure end doesn't exceed the limit
    yield list(range(start, end))
    start = end

def get_total_iterations(limit, chunk_size):
  """Calculates the total number of iterations."""
  return (limit + chunk_size - 1) // chunk_size

total_iterations = get_total_iterations(limit, chunk_size)

# Example usage:
for indices in tqdm(generate_indices(limit, chunk_size), total=total_iterations):
    try:
        results = asyncio.run(get_diagnoses(indices))
        for result in results:
          hadm_id = result[0]
          content = result[1]
          result_df.loc[hadm_id, 'GPT-Diagnoses'] = content
    except Exception as e:
        print('Error happened at iteration i: ' + str(indices[0]/chunk_size))
        repeat.append(i for i in indices)
        print(e)

100%|██████████| 200/200 [18:12<00:00,  5.46s/it]


In [ ]:
result_df.to_csv('/content/drive/MyDrive/LLaMa4ScoutProdDiagnoses.csv')

In [ ]:
result_df.loc[2].GPT_input[-750:]

'een 3 and 174 hours after admission the patient stayed in the ICU and during this period had the following measurements min heart rate: 49, max heart rate: 103, avg heart rate: 75.19, min systolic blood pressure: 87, max systolic blood pressure: 141, avg systolic blood pressure: 108.76, min diastolic blood pressure: 39, max diastolic blood pressure: 82, avg diastolic blood pressure: 60.34, min respiration rate: 9, max respiration rate: 34, avg respiration rate: 17, min temperature: 35.83, max temperature: 37.94, avg temperature: 36.73, min peripheral oxygen saturation: 88, max peripheral oxygen saturation: 100, avg peripheral oxygen saturation: 95.21, min blood glucose level: 79, max blood glucose level: 167, avg blood glucose level: 115.88'

In [ ]:
result_df.loc[2]['GPT-Diagnoses']

'*Respiratory Acidosis*\n• pco2 range: 37-55 \n• pH range: 7.33-7.46 \n• Base excess range: -1 to 6 \n• Alternative differentials: Metabolic alkalosis, Mixed acid-base disorder\n\n*Respiratory Failure*\n• Min PO2: 63 \n• Min peripheral oxygen saturation: 88 \n• Max fio2: 40 \n• PEEP: 5 \n• Alternative differentials: Cardiac failure, Anemia \n\n*Anemia*\n• Min hemoglobin (g/dl): 10.1 \n• Min hematocrit: 27.9 \n• Alternative differentials: Blood loss, Bone marrow failure \n\n*Infection/Inflammation*\n• Max WBC: 19.3 \n• Avg WBC: 13.11 \n• Alternative differentials: Trauma, Surgery \n\n*Pleural Effusion*\n• Imaging findings of pleural effusion \n• Alternative differentials: Pulmonary embolism, Heart failure \n\n*Pneumomediastinum*\n• Imaging findings of pneumomediastinum \n• Alternative differentials: Pneumothorax, Esophageal rupture \n\n*Post-operative complications*\n• History of mitral valve surgery \n• Imaging findings of atelectasis and pleural effusion \n• Alternative differentials: